# Week 5 — `min`, `max`, and `concat`

Each Maddison file in `week-05/materials/maddison_by_country/` is one entity, with columns `Entity`, `Code`, `Year`, and `GDP`.

`.min()` and `.max()` reduce a column to one value. `pd.concat` stacks those files back into one table.

Run this notebook from `introduction-to-data-engineering`, from `week-05/materials`, or from `week-05/student_work`.


## Find the folder


In [38]:
from pathlib import Path

import pandas as pd


def find_existing(candidates):
    for path in candidates:
        if path.exists():
            return path
    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find the Maddison folder. Looked in:\n" + searched
    )


cwd = Path.cwd()
maddison_dir = find_existing(
    [
        cwd / "maddison_by_country",
    ]
)
maddison_dir


PosixPath('/Users/apple/SourceCode/DEAI-Introduction-to-Data-Engineering/week-05/maddison_by_country')

## One file: Finland

`.min()` is the smallest value in that column. `.max()` is the largest. `Year.min()` is the earliest year in the file. `Year.max()` is the latest year.


In [39]:
finland = pd.read_csv(maddison_dir / "Finland.csv")
finland.head(3)


,Entity,Code,Year,GDP,GDP (Annotations)
0,Finland,FIN,1600,476400000,NaN
1,Finland,FIN,1700,532400000,NaN
2,Finland,FIN,1820,1534897000,NaN


In [40]:
finland["Year"].min(), finland["Year"].max()


(np.int64(1600), np.int64(2022))

In [41]:
finland["GDP"].min(), finland["GDP"].max()


(np.int64(476400000), np.int64(226154847456))

The number alone leaves out the year it belongs to. `.idxmin()` is the row label where GDP is smallest. `.loc` reads that row. `.idxmax()` does the same for the largest GDP.


In [42]:
finland.loc[finland["GDP"].idxmin(), ["Entity", "Year", "GDP"]]


Entity      Finland
Year           1600
GDP       476400000
Name: 0, dtype: object

In [43]:
finland.loc[finland["GDP"].idxmax(), ["Entity", "Year", "GDP"]]


Entity         Finland
Year              2022
GDP       226154847456
Name: 166, dtype: object

Finland's file runs from 1600 to 2022. The smallest GDP is in 1600, and the largest GDP is in 2022. On this file the two year questions happen to land on the same rows.


## The two year questions on Afghanistan

`Year.min()` answers "what is the earliest year in these rows?".

`df.loc[df["GDP"].idxmin(), "Year"]` answers "in which year is GDP smallest?".

On Afghanistan those years differ. Run both before you decide which question you meant.


In [44]:
afghanistan = pd.read_csv(maddison_dir / "Afghanistan.csv")
afghanistan["Year"].min(), afghanistan["Year"].max()


(np.int64(1950), np.int64(2022))

In [45]:
afghanistan.loc[afghanistan["GDP"].idxmin(), ["Entity", "Year", "GDP"]]


Entity    Afghanistan
Year             1994
GDP        7919856426
Name: 44, dtype: object

In [46]:
afghanistan.loc[afghanistan["GDP"].idxmax(), ["Entity", "Year", "GDP"]]


Entity    Afghanistan
Year             2019
GDP       73085701943
Name: 69, dtype: object

The Afghanistan file starts in 1950 and ends in 2022. The smallest GDP is in 1994. The largest GDP is in 2019.


## Stack two files with `concat`

Build a list of DataFrames, then stack them. `ignore_index=True` writes a new row index. The row count equals the sum of the two files.


In [47]:
finland = pd.read_csv(maddison_dir / "Finland.csv")
sweden = pd.read_csv(maddison_dir / "Sweden.csv")
frames = [finland, sweden]
two_countries = pd.concat(frames, ignore_index=True)
len(finland), len(sweden), len(two_countries)


(167, 206, 373)

In [48]:
len(two_countries) == len(finland) + len(sweden)


True

In [49]:
two_countries["Entity"].unique()


<StringArray>
['Finland', 'Sweden']
Length: 2, dtype: str

## In depth: stack every file

The loop reads each CSV into a list. `concat` stacks the list. One entity per file means `nunique()` matches the number of files.

Compare Finland inside the stacked table with `Finland.csv` on `Entity`, `Year`, and `GDP`.


In [50]:
all_frames = []
for path in sorted(maddison_dir.glob("*.csv")):
    all_frames.append(pd.read_csv(path))

df = pd.concat(all_frames, ignore_index=True)
len(all_frames), df["Entity"].nunique(), len(df)


(178, 178, 16143)

In [51]:
len(df) == sum(len(frame) for frame in all_frames)


True

In [52]:
columns = ["Entity", "Year", "GDP"]
from_file = finland[columns].reset_index(drop=True)
from_all = df.loc[df["Entity"] == "Finland", columns].reset_index(drop=True)
from_file.equals(from_all)


True

## Practice

1. Read `Sweden.csv`. Report the earliest year, the latest year, the smallest GDP, the largest GDP, and the year of the smallest GDP.
2. You already stacked Finland and Sweden. What is `len(two_countries)`, and which two names are in `Entity`?
3. From the full table `df`, show the Afghanistan row where GDP is largest. Keep `Entity`, `Year`, and `GDP`.


In [53]:
# Practice 1
sweden = pd.read_csv(maddison_dir / "Sweden.csv")

print(f"Earlier years of Sweden: {sweden['Year'].min()}")
print(f"Latest years of Sweden: {sweden['Year'].max()}")
print(f"Smallest GDP of Sweden: {sweden['GDP'].min()}")
print(f"Latest GDP of Sweden: {sweden['GDP'].max()}")

Earlier years of Sweden: 1500
Latest years of Sweden: 2022
Smallest GDP of Sweden: 956080000
Latest GDP of Sweden: 494214521158


In [54]:
# Practice 2
two_countries = pd.concat([finland, sweden], ignore_index=True)
print(f"Len of the two countries is {len(two_countries)}")
print(f"The 2 names of the 2 countries are {two_countries['Entity'].unique()}")

Len of the two countries is 373
The 2 names of the 2 countries are <StringArray>
['Finland', 'Sweden']
Length: 2, dtype: str


In [55]:
# Practice 3
# This is the best optimization code! 
# Filtering the dataframe to only include Afghanistan data and then finding the row with the maximum GDP 
# is more efficient than filtering the entire dataframe multiple times.
afghanistan_data = df.loc[df["Entity"] == "Afghanistan", ["Year", "GDP"]]
afghanistan_largest_row = afghanistan_data.loc[afghanistan_data["GDP"].idxmax()]
afghanistan_largest_row

Year           2019
GDP     73085701943
Name: 69, dtype: int64

## Quiz

**1.** What does `finland["Year"].min()` return?

- A) The whole first row
- B) The earliest year in that column --> True
- C) The year in which GDP is smallest
- D) The number of rows

**2.** Which line returns the year in which GDP is smallest?

- A) `df["Year"].min()`
- B) `df["GDP"].min()`
- C) `df.loc[df["GDP"].idxmin(), "Year"]` --> True
- D) `df["Year"].max()`

**3.** `pd.concat([finland, sweden], ignore_index=True)` produces a table whose row count is:

- A) The larger of the two files
- B) The sum of the two files' row counts --> True
- C) The number of columns
- D) One row per year that appears in both files

**4.** On Afghanistan, `Year.min()` is 1950. The year of the smallest GDP is:

- A) 1950
- B) 1994 --> True
- C) 2019
- D) 2022
